# External Data Downloads (Q2 2025, Daily)

Each download is executed in its own independent cell. The only shared cells are the setup utilities at the top.


In [ ]:
# General setup
# - Use absolute paths only to avoid writing files into unexpected folders.
# - Keep the date window aligned with the daily sales target.

from __future__ import annotations

import subprocess
import sys
from pathlib import Path

BASE_DIR = Path(r"C:/Users/simon/food_prediction/raw_data/code_external_data")
REPO_ROOT = Path(r"C:/Users/simon/food_prediction")
OUT_DIR = (BASE_DIR / "_external_data").resolve()
STORES_FILE = (REPO_ROOT / "raw_data" / "20260218_144523_stores.parquet").resolve()
PLZ_CENTROIDS_FILE = (BASE_DIR / "_reference_geo" / "plz_centroids.csv").resolve()

START = "2025-04-01"
END = "2025-06-30"

OUT_DIR.mkdir(parents=True, exist_ok=True)

def run_script(script_name: str, args: list[str]) -> None:
    """Run a downloader script via the current Python interpreter.

    We call scripts through sys.executable so the notebook uses the same venv
    where pandas/pyarrow are installed.
    """
    script_path = (BASE_DIR / script_name).resolve()
    cmd = [sys.executable, str(script_path), *args]
    print("\n\u25B6", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
# Download German public holidays (incl. NRW flag) from Nager.Date.
# Output: one table with 'date' and holiday meta fields (daily).

run_script(
    "download_01_holidays_nagerdate.py",
    ["--start", START, "--end", END, "--out-dir", str(OUT_DIR), "--force"],
)


In [ ]:
# Download NRW school holidays (ranges + daily flag table) from ferien-api.de.

run_script(
    "download_02_school_holidays_ferien_api.py",
    ["--start", START, "--end", END, "--state", "NW", "--out-dir", str(OUT_DIR), "--force"],
)


In [ ]:
# Download hourly weather from Bright Sky (DWD-based) and aggregate to daily per store.
# Stores have ZIP but no coordinates, so we derive lat/lon via PLZ centroids.

run_script(
    "download_07_dwd_weather_brightsky.py",
    ["--start", START, "--end", END, "--stores-file", str(STORES_FILE), "--plz-centroids-file", str(PLZ_CENTROIDS_FILE), "--out-dir", str(OUT_DIR), "--force"],
)


In [ ]:
# Download OSM POI counts around each store (static store-level).
# This returns aggregated counts only (no geometries) to keep disk/RAM usage low.

run_script(
    "download_03_osm_pois_overpass.py",
    ["--stores-file", str(STORES_FILE), "--plz-centroids-file", str(PLZ_CENTROIDS_FILE), "--radius-m", "1000", "--out-dir", str(OUT_DIR), "--force"],
)


In [ ]:
# Download air quality measurements (UBA) and aggregate to daily per store.
# This can be slower than the other sources due to API calls per station/component.

run_script(
    "download_04_uba_air_quality.py",
    ["--start", START, "--end", END, "--stores-file", str(STORES_FILE), "--plz-centroids-file", str(PLZ_CENTROIDS_FILE), "--max-distance-km", "20", "--out-dir", str(OUT_DIR), "--force"],
)


In [ ]:
# Download football match schedule from OpenLigaDB and aggregate to daily tables.
# The provider's notion of 'season' can differ; the script will auto-try adjacent seasons.

run_script(
    "download_05_openligadb_matches.py",
    ["--start", START, "--end", END, "--league", "bl1", "--season", "2024", "--out-dir", str(OUT_DIR), "--force"],
)
